# EDA — BADESTELLENOGD.csv (Badestellen / bathing sites)

City of Vienna open data export. Source file: `data/raw/BADESTELLENOGD.csv`. Run all cells to
reproduce the findings below.

In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/BADESTELLENOGD.csv")
df.shape

(32, 14)

## Columns & dtypes

In [2]:
df.dtypes

FID                       str
SHAPE                     str
BEZEICHNUNG               str
WEITERE_INFO              str
BEZIRK                float64
BADEQUALITAET           int64
UNTERSUCHUNGSDATUM        str
WASSERTEMPERATUR      float64
SICHTTIEFE            float64
ANZ_ECOLI               int64
ANZ_ENTEROKOKKEN        int64
TYP                     int64
SE_SDO_ROWID            int64
SE_ANNO_CAD_DATA      float64
dtype: object

## Missing values

In [3]:
df.isnull().sum()

FID                    0
SHAPE                  0
BEZEICHNUNG            0
WEITERE_INFO           0
BEZIRK                 2
BADEQUALITAET          0
UNTERSUCHUNGSDATUM     0
WASSERTEMPERATUR       0
SICHTTIEFE             2
ANZ_ECOLI              0
ANZ_ENTEROKOKKEN       0
TYP                    0
SE_SDO_ROWID           0
SE_ANNO_CAD_DATA      32
dtype: int64

## Duplicate check

In [4]:
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate BEZEICHNUNG:", df["BEZEICHNUNG"].duplicated().sum())

Duplicate full rows: 0
Duplicate BEZEICHNUNG: 0


## Parse geometry

`SHAPE` is a WKT geometry string. Parse into `lon`/`lat` (works for the first
coordinate pair even if the geometry is a line/polygon) and sanity-check the
range against Vienna's bounding box.

In [5]:
def parse_first_point(s):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(s))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

df["lon"], df["lat"] = zip(*df["SHAPE"].map(parse_first_point))
print("Unparseable SHAPE values:", df["lon"].isnull().sum())
print("lon range:", df["lon"].min(), "-", df["lon"].max())
print("lat range:", df["lat"].min(), "-", df["lat"].max())

Unparseable SHAPE values: 0
lon range: 16.34941579686922 - 16.541003469998593
lat range: 48.167049590267446 - 48.283555826709474


## District (`BEZIRK`) distribution

In [6]:
df["BEZIRK"].value_counts(dropna=False).sort_index()

BEZIRK
21.0     5
22.0    25
NaN      2
Name: count, dtype: int64

## Bathing-site-specific fields

`BADEQUALITAET` and `TYP` are numeric codes with no legend in this file — the
values themselves are visible below, but their meaning needs to come from the
dataset's metadata page on data.gv.at, not guessed here. `UNTERSUCHUNGSDATUM`
(water-quality test date) looks like it's kept current, not a one-time static
value — worth checking against today's date.

In [7]:
print("BADEQUALITAET value counts:")
print(df["BADEQUALITAET"].value_counts(dropna=False))
print("\nTYP value counts:")
print(df["TYP"].value_counts(dropna=False))
print("\nUNTERSUCHUNGSDATUM sample (most recent test dates):")
print(df["UNTERSUCHUNGSDATUM"].sort_values(ascending=False).head(5).tolist())

BADEQUALITAET value counts:
BADEQUALITAET
1    29
2     3
Name: count, dtype: int64

TYP value counts:
TYP
1    17
2    15
Name: count, dtype: int64

UNTERSUCHUNGSDATUM sample (most recent test dates):
['2026-08-10', '2026-08-10', '2026-08-10', '2026-08-10', '2026-08-10']


## Sample rows

In [8]:
df[["BEZEICHNUNG", "BEZIRK", "TYP", "BADEQUALITAET", "lon", "lat"]].sample(5, random_state=1)

,BEZEICHNUNG,BEZIRK,TYP,BADEQUALITAET,lon,lat
27,Teich Hirschstetten,22.0,2,2,16.478820,48.243925
3,"Neue Donau stromab Reichsbrücke, links",22.0,1,1,16.416656,48.226215
22,Asperner See,22.0,2,1,16.506354,48.228015
18,Alte Naufahrt,22.0,2,1,16.474811,48.195439
23,"Neue Donau, Familienbadestrand",NaN,2,2,16.396636,48.243099


## Findings

**Basics:** 32 records, 14 columns. Small, clean dataset.

**Missing values:** `BEZIRK` and `SICHTTIEFE` (visibility depth) each missing for
2 rows — likely bathing sites that span/don't cleanly belong to one district
(e.g. along the Alte/Neue Donau), or weren't part of the latest visibility survey.

**No duplicates** — neither full-row nor on `BEZEICHNUNG`.

**Geometry:** clean WGS84 points, all parse successfully.

**Notable:** `UNTERSUCHUNGSDATUM` (water quality test date) contains recent dates —
this "static" export is actually refreshed regularly with water-quality test
results, similar in spirit to the live occupancy data in `SCHWIMMBADOGD.csv`. Could
be a second temporal/live signal for the reasoning layer (e.g. "only suggest
bathing sites with good recent water quality").

**Open question:** `BADEQUALITAET` (2 distinct values, e.g. 1/2) and `TYP` (also
2 distinct values) are undocumented numeric codes — need the data.gv.at metadata
page to interpret them correctly before using in the KG (don't guess at meaning).

**Suitability for the KG:** good candidate. Suggested mapping: `BEZEICHNUNG` →
`poi:name`, `BEZIRK` → `poi:district`, `lon`/`lat` → `geo:long`/`geo:lat`,
`BADEQUALITAET`/`TYP` → keep as coded attributes pending legend, `UNTERSUCHUNGSDATUM`
→ a temporal property if the live-quality angle gets pursued.